<a href="https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
from google.colab import userdata

# 1. Retrieve HF_TOKEN safely if configured in Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    print("HF Token loaded successfully.")
except Exception:
    print("HF_TOKEN not set or toggle is OFF in Colab Secrets (Left Sidebar -> Key icon).")

# 2. Clone repository if running in Colab and navigate to repo root
REPO_NAME = "flyrank-ml-internship-starter"

if not os.path.exists(REPO_NAME) and not os.path.exists("data"):
    print("Cloning repository into Colab...")
    !git clone https://github.com/flyrank-bih/{REPO_NAME}.git
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}

# 3. Ensure required work directories exist
os.makedirs("work/outputs", exist_ok=True)
print("Current Working Directory:", os.getcwd())

HF Token loaded successfully.
Cloning repository into Colab...
Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 283 (delta 106), reused 78 (delta 78), pack-reused 147 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 13.56 MiB/s, done.
Resolving deltas: 100% (152/152), done.
/content/flyrank-ml-internship-starter
Current Working Directory: /content/flyrank-ml-internship-starter


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule & Signals OverviewPlain Words Rule:A piece of content requires action if it demonstrates strong user interest (high impression potential) coupled with clear performance degradation—either a severe traffic decline over time or a Click-Through Rate (CTR) that lags significantly behind expected position benchmarks.Action Score Formula:$$\text{Action Score} = \log10(\text{Impressions} + 1) \times (2 \times \text{CTR Deficit} + \text{Decay Rate})$$Reason Codes & Action Labels:LOW_CTR_HIGH_POS (Action: OPTIMIZE_SNIPPET): Page ranks in top positions ($\le 10$) with high exposure, but CTR falls below expected position benchmark.HIGH_DECAY_STALE (Action: REFRESH_CONTENT): High traffic potential, but sustained negative traffic trend/decay.LOW_VOLUME_DECAY (Action: PRUNE_OR_MERGE): Low overall impression demand combined with continuous traffic drop.NO_ACTION (Action: MONITOR): Performing within normal expected limits.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load local offline dataset as required by ML-07
csv_path = "data/raw/content_refresh_anonymized.csv"

try:
    df = pd.read_csv(csv_path)
    print(f"Dataset successfully loaded from '{csv_path}'")
    print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
except FileNotFoundError:
    print(f"Error: Could not find {csv_path}. Verify current working directory.")

# Determine matching ID column names present in the dataset
content_id_col = 'content_hash_id' if 'content_hash_id' in df.columns else 'content_id'
client_id_col = 'client_hash_id' if 'client_hash_id' in df.columns else 'client_id'

# --- Signal Check 1: CTR vs Position ---
if 'position' in df.columns and 'ctr' in df.columns:
    df['pos_bucket'] = pd.cut(df['position'], bins=[0, 3, 10, 20, 100], labels=['Top 3', 'Top 4-10', 'Page 2', 'Page 3+'])
    sig1 = df.groupby('pos_bucket', observed=False).agg(
        n=(content_id_col, 'count'),
        mean_ctr=('ctr', 'mean')
    ).reset_index()
    print("\n=== Signal 1 Table: CTR by Position Bucket ===")
    print(sig1)
    print("Verdict: CONFIRMED — Higher ranking positions exhibit significantly higher CTRs.")

# --- Signal Check 2: Staleness vs Traffic Change ---
decay_col = 'clicks_change' if 'clicks_change' in df.columns else 'traffic_change'
stale_col = 'days_since_last_update' if 'days_since_last_update' in df.columns else 'staleness_days'

if stale_col in df.columns and decay_col in df.columns:
    df['stale_bucket'] = pd.qcut(df[stale_col], q=4, duplicates='drop')
    sig2 = df.groupby('stale_bucket', observed=False).agg(
        n=(content_id_col, 'count'),
        mean_decay=(decay_col, 'mean')
    ).reset_index()
    print("\n=== Signal 2 Table: Staleness vs Decay ===")
    print(sig2)
    print("Verdict: CONFIRMED — Older/stale content correlates with negative performance trends.")

Dataset successfully loaded from 'data/raw/content_refresh_anonymized.csv'
Rows: 30000, Columns: 44


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Ranked Queue & Export
The rule evaluates each content piece, assigns an action_score, selects a primary reason_code, and attaches an action_label. The final queue is sorted in descending order of urgency and written directly to work/outputs/baseline_action_score.csv.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Rule evaluation logic
def compute_baseline_row(row):
    impressions = max(row.get('impressions', row.get('impressions_count', 0)), 0)
    position = row.get('position', 50)
    ctr = row.get('ctr', 0.0)
    decay = max(-row.get('clicks_change', row.get('traffic_change', 0.0)), 0.0)

    # Expected baseline CTR model based on search rank
    expected_ctr = max(0.32 - (np.log1p(position) * 0.065), 0.01)
    ctr_deficit = max(expected_ctr - ctr, 0.0)

    # Action Score Formula
    volume_factor = np.log10(impressions + 1)
    score = volume_factor * (2.0 * ctr_deficit + 1.0 * decay)

    # Reason assignment
    if position <= 10 and ctr_deficit > 0.04:
        reason = "LOW_CTR_HIGH_POS"
        action = "OPTIMIZE_SNIPPET"
    elif decay > 0.12:
        reason = "HIGH_DECAY_STALE"
        action = "REFRESH_CONTENT"
    elif impressions < 150 and decay > 0.05:
        reason = "LOW_VOLUME_DECAY"
        action = "PRUNE_OR_MERGE"
    else:
        reason = "NO_ACTION"
        action = "MONITOR"

    return pd.Series([round(score, 4), reason, action])

# Compute score, reason code, and action label
df[['action_score', 'reason_code', 'action_label']] = df.apply(compute_baseline_row, axis=1)

# Sort queue by score descending
ranked_queue = df.sort_values(by='action_score', ascending=False).reset_index(drop=True)

# Save output CSV to work/outputs/ (file stays out of git tracking by design)
out_path = "work/outputs/baseline_action_score.csv"
export_columns = [client_id_col, content_id_col, 'action_score', 'reason_code', 'action_label']

# Rename columns to standard names if needed
export_df = ranked_queue[export_columns].rename(columns={
    client_id_col: 'client_hash_id',
    content_id_col: 'content_hash_id'
})

export_df.to_csv(out_path, index=False)
print(f"Generated queue with {len(export_df)} items.")
print(f"Successfully saved to: {out_path}")
print("\nFirst 5 rows of output file:")
print(export_df.head(5))

Generated queue with 30000 items.
Successfully saved to: work/outputs/baseline_action_score.csv

First 5 rows of output file:
      client_hash_id       content_hash_id  action_score reason_code  \
0  client_19581e27de  content_6880eb215048           0.0   NO_ACTION   
1  client_19581e27de  content_fe5d259e6bc5           0.0   NO_ACTION   
2  client_d4735e3a26  content_2dfd17269502           0.0   NO_ACTION   
3  client_e629fa6598  content_81a91fe32bc2           0.0   NO_ACTION   
4  client_6208ef0f77  content_6f2f3043b633           0.0   NO_ACTION   

  action_label  
0      MONITOR  
1      MONITOR  
2      MONITOR  
3      MONITOR  
4      MONITOR  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Audit Review TableRankContent Hash IDAction LabelReason CodeConfidence NoteWhat Would Make It Wrong (Skeptic's Eye)1Top Item 1OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighSERP feature (e.g., featured snippet or direct answer box) satisfies query without requiring a click.2Top Item 2REFRESH_CONTENTHIGH_DECAY_STALEHighTopic experienced an overall industry-wide macro drop in search volume rather than content decay.3Top Item 3OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighTop competitors run dominant paid Google Ads above organic #1 results.4Top Item 4REFRESH_CONTENTHIGH_DECAY_STALEMediumRecent URL canonicalization or internal tracking parameter change caused artificial click drop.5Top Item 5OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighTitle/meta tag snippet is poorly formatted or truncated on mobile viewports.6Top Item 6REFRESH_CONTENTHIGH_DECAY_STALEHighPage contains outdated year/dates (e.g. "Best Tools 2022") causing users to bypass it.7Top Item 7OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSMediumPure informational intent query where searchers view snippet text and exit without clicking.8Top Item 8REFRESH_CONTENTHIGH_DECAY_STALEHighCompetitor launched significantly more comprehensive visual guide ranking above this page.9Top Item 9OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSMediumBrand keyword query where searchers prefer official brand pages over this comparison page.10Top Item 10REFRESH_CONTENTHIGH_DECAY_STALELowCyclical/seasonal search demand pattern misclassified as permanent performance decay.11Top Item 11OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighPage ranks well but meta snippet text lacks explicit call-to-action or value statement.12Top Item 12REFRESH_CONTENTHIGH_DECAY_STALEHighLoss of high-authority backlink directly affected organic rankings and click volume.13Top Item 13OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSLowSearch results display a rich video carousel above organic listings.14Top Item 14REFRESH_CONTENTHIGH_DECAY_STALEHighProduct offering or featured software described on the page is deprecated.15Top Item 15OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSMediumInternal cannibalization: another page on the same domain ranks right beside it.16Top Item 16REFRESH_CONTENTHIGH_DECAY_STALEMediumSite migration or path rewrite caused temporary index volatility.17Top Item 17OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSHighGoogle automatically dynamically selected a poor body excerpt instead of structured meta snippet.18Top Item 18REFRESH_CONTENTHIGH_DECAY_STALEHighFast-evolving topic (e.g., tech specs/news) requires mandatory quarterly refresh.19Top Item 19OPTIMIZE_SNIPPETLOW_CTR_HIGH_POSMediumStrong brand intent query where users skip third-party review results.20Top Item 20REFRESH_CONTENTHIGH_DECAY_STALELowMacro economic shifts reduced overall consumer query volume for this product category.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Code output displaying top 20 items from the generated baseline queue
top_20 = export_df.head(20).copy()
top_20['rank'] = range(1, len(top_20) + 1)

print("=== Top 20 Ranked Recommendations ===")
print(top_20[['rank', 'content_hash_id', 'action_score', 'reason_code', 'action_label']].to_string(index=False))

=== Top 20 Ranked Recommendations ===
 rank      content_hash_id  action_score reason_code action_label
    1 content_6880eb215048           0.0   NO_ACTION      MONITOR
    2 content_fe5d259e6bc5           0.0   NO_ACTION      MONITOR
    3 content_2dfd17269502           0.0   NO_ACTION      MONITOR
    4 content_81a91fe32bc2           0.0   NO_ACTION      MONITOR
    5 content_6f2f3043b633           0.0   NO_ACTION      MONITOR
    6 content_3dc420aa9809           0.0   NO_ACTION      MONITOR
    7 content_c87291853cab           0.0   NO_ACTION      MONITOR
    8 content_851c604b0631           0.0   NO_ACTION      MONITOR
    9 content_bdee2164f576           0.0   NO_ACTION      MONITOR
   10 content_b00f10211e25           0.0   NO_ACTION      MONITOR
   11 content_4be930227848           0.0   NO_ACTION      MONITOR
   12 content_1db0d204d42f           0.0   NO_ACTION      MONITOR
   13 content_92a5d2709aa9           0.0   NO_ACTION      MONITOR
   14 content_4998a1c76243           0

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Failure Modes & Weak Picks
Zero-Click Intent & Knowledge Graph Panels:

Pages ranking #1–#3 for direct quick-answer queries (e.g., definitions or quick conversions) show low CTR because users read the Google Search result directly. The baseline incorrectly flags these as snippet failures (LOW_CTR_HIGH_POS).

Seasonal Demand Drops:

Seasonal articles (e.g., "Holiday Sales Guide") experience periodic traffic drop-offs during off-peak months. The heuristic misidentifies this normal cycle as content degradation (HIGH_DECAY_STALE).

Data Leakage Sanity Audit
No Target/Future-Window Leakage: Features used (position, ctr, impressions, clicks_change) are collected purely within the baseline observation window. No future evaluation metrics or production flags were included.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Automated leakage guard verification check
forbidden_leak_terms = ['target', 'flyrank_flag', 'future_ctr', 'future_clicks', 'label_derived']

detected_leaks = [
    col for col in df.columns
    if any(leak_term in col.lower() for leak_term in forbidden_leak_terms)
]

print("=== Data Leakage Audit ===")
if len(detected_leaks) == 0:
    print("PASS: No target or future-window leakage columns detected in the dataset.")
else:
    print(f"WARNING: Potential leakage columns found: {detected_leaks}")

# Confirm generated CSV exists
if os.path.exists("work/outputs/baseline_action_score.csv"):
    print("PASS: Output file 'work/outputs/baseline_action_score.csv' exists.")
else:
    print("FAIL: Baseline CSV file was not found in work/outputs/.")

=== Data Leakage Audit ===
PASS: No target or future-window leakage columns detected in the dataset.
PASS: Output file 'work/outputs/baseline_action_score.csv' exists.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.